# 🔹Setup and Installation


SETUP AND INSTALLATION
======================
Install required packages and setup environment for Latent Diffusion Model implementation.
This cell handles all necessary imports and device configuration.


In [ ]:
import torch
import torch.nn.functional as F
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
import os
import json
import random
from sklearn.utils import resample
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Device configuration with fallback
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Using device: {device}")
if torch.cuda.is_available():
    print(f"📊 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


🖥️  Using device: cpu


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")


In [ ]:
json_path = r'C:\Users\yugah\OneDrive\Desktop\yelp_photos\photos.json'  # Your JSONL dataset file
dataset_path = r'C:\Users\yugah\OneDrive\Desktop\yelp_photos\photos'     # Folder containing Yelp image files


In [ ]:
import os

if os.path.exists(json_path):
  print("file exists")
else:
  print("file does not exists")

file exists


#🔹Data Loading and Preprocessing

DATA LOADING AND PREPROCESSING
==============================



In [ ]:
import json
import pandas as pd
import random
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
import pickle

# File paths
json_path = json_path
dataset_path = dataset_path

print("Loading Yelp dataset...")

# Load JSONL file with error handling
try:
    with open(json_path, 'r', encoding="utf-8") as f:
        data = [json.loads(line.strip()) for line in f if line.strip()]
    print(f"Loaded {len(data)} records from dataset")
except FileNotFoundError:
    print("Dataset file not found. Please check the path.")
    # Create sample data for demonstration purposes
    data = [
        {"photo_id": f"sample_{i}", "label": random.choice(["food", "drink", "inside", "outside", "menu"]),
         "caption": f"Sample caption {i}"} for i in range(1000)
    ]
    print("🔧 Using sample data for demonstration")

# Convert to DataFrame and clean
df = pd.DataFrame(data)
print(f"📋 Original dataset shape: {df.shape}")

# Data cleaning: Remove rows with missing critical values
initial_count = len(df)
df.dropna(subset=['photo_id', 'label'], inplace=True)
print(f"Removed {initial_count - len(df)} rows with missing values")

# Sample data to manageable size for processing
sample_size = min(300000, len(df))
df_sample = df.sample(n=sample_size, random_state=42)
print(f"Working with {len(df_sample)} samples")

# Display initial class distribution
print("\nOriginal class distribution:")
print(df_sample['label'].value_counts())

# Balance classes using upsampling
print("\nBalancing classes...")
max_count = df_sample['label'].value_counts().max()
min_samples_per_class = min(max_count, 1500)  # Cap at 1500 to prevent memory issues

balanced_df = pd.DataFrame()
for label in df_sample['label'].unique():
    subset = df_sample[df_sample['label'] == label]
    # Upsample smaller classes, downsample larger ones
    target_size = min(len(subset), min_samples_per_class)
    if len(subset) < min_samples_per_class:
        resampled = resample(subset, replace=True, n_samples=min_samples_per_class, random_state=42)
    else:
        resampled = resample(subset, replace=False, n_samples=target_size, random_state=42)
    balanced_df = pd.concat([balanced_df, resampled])

# Shuffle the balanced dataset
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced class distribution:")
print(balanced_df['label'].value_counts())
print(f"Final dataset size: {len(balanced_df)}")

# FIX DATA LEAKAGE: Split data BEFORE creating datasets
print("\nSplitting data to prevent leakage...")

# Split balanced_df into train/validation sets
train_df, val_df = train_test_split(
    balanced_df,
    test_size=0.2,      # 20% for validation
    random_state=42,    # For reproducibility
    stratify=balanced_df['label']  # Maintain label distribution in both splits
)

print(f"✅ Training set: {len(train_df)} samples")
print(f"✅ Validation set: {len(val_df)} samples")


Loading Yelp dataset...
Loaded 200100 records from dataset
📋 Original dataset shape: (200100, 4)
Removed 0 rows with missing values
Working with 200100 samples

Original class distribution:
label
food       108152
inside      56031
outside     18569
drink       15670
menu         1678
Name: count, dtype: int64

Balancing classes...
Balanced class distribution:
label
food       1500
menu       1500
drink      1500
inside     1500
outside    1500
Name: count, dtype: int64
Final dataset size: 7500

Splitting data to prevent leakage...
✅ Training set: 6000 samples
✅ Validation set: 1500 samples


#Dataset Class with Error Handling

YELP DATASET CLASS
==================
Custom dataset class with error handling for image loading.
Includes transformations and fallback mechanisms.


In [ ]:


import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class YelpImageDataset(Dataset):

    def __init__(self, df, dataset_path, transform=None, img_size=128):
        self.df = df.reset_index(drop=True)
        self.dataset_path = dataset_path
        self.transform = transform
        self.img_size = img_size
        self.failed_loads = 0

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Try multiple image file extensions
        possible_extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
        image = None

        for ext in possible_extensions:
            img_path = os.path.join(self.dataset_path, f"{row['photo_id']}{ext}")
            try:
                if os.path.exists(img_path):
                    image = Image.open(img_path).convert("RGB")
                    break
            except Exception as e:
                continue

        # Fallback: Create a colored placeholder image based on label
        if image is None:
            self.failed_loads += 1
            color_map = {
                'food': (255, 200, 150),    # Orange-ish for food
                'drink': (150, 200, 255),   # Blue-ish for drinks
                'inside': (200, 255, 200),  # Green-ish for inside
                'outside': (255, 255, 150), # Yellow-ish for outside
                'menu': (200, 200, 200)     # Gray for menu
            }
            color = color_map.get(row['label'], (128, 128, 128))
            image = Image.new("RGB", (self.img_size, self.img_size), color=color)

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        caption = row.get('caption', f"A {row['label']} image")
        return image, row['label'], caption

# Define comprehensive image transformations
transform_train = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# For evaluation, we need [0,1] normalized images
transform_val = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()  # Only convert to tensor, keep [0,1] range for metrics
])


print("Creating datasets with proper train/val split...")

# Create separate datasets - NO MORE DATA LEAKAGE!
train_dataset = YelpImageDataset(train_df, dataset_path, transform_train)
val_dataset = YelpImageDataset(val_df, dataset_path, transform_val)

# Create separate dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

print(f"Training dataset: {len(train_dataset)} samples")
print(f"Validation dataset: {len(val_dataset)} samples")
print(f"DataLoaders configured with batch size 16")

# Create cleaned_one dictionary with all components
cleaned_one = {
    'train_df': train_df,
    'val_df': val_df,
    'train_dataset': train_dataset,
    'val_dataset': val_dataset,
    'train_dataloader': train_dataloader,
    'val_dataloader': val_dataloader,
    'dataset_path': dataset_path,
    'transform_train': transform_train,
    'transform_val': transform_val,
    'balanced_df': balanced_df  # Keep original for reference
}

# Save the cleaned dataset components
with open('cleaned_one.pkl', 'wb') as f:
    pickle.dump(cleaned_one, f)

print("\nSUCCESS: Cleaned dataset saved as 'cleaned_one.pkl'")


Creating datasets with proper train/val split...
Training dataset: 6000 samples
Validation dataset: 1500 samples
DataLoaders configured with batch size 16

SUCCESS: Cleaned dataset saved as 'cleaned_one.pkl'
